In [1]:
import pandas as pd
from pathlib import Path

# Ordner mit unseren Kursdateien
ordner = Path.home() / "Desktop" / "Modul 5" / "Aufgaben" / "Woche_2"

# CSV-Datei einlesen
datei = ordner / "bestellungen_q1_roh.csv"

df = pd.read_csv(datei, sep=None, engine="python")

# Erste Übersicht
print("Zeilen und Spalten:", df.shape)
display(df.head())

Zeilen und Spalten: (218, 9)


,bestell_id,datum,kunde_email,produkt,kategorie,menge,preis_eur,land,status
0,3101,2026-02-21,kontakt-fehlt,Dockingstation,Zubehör,2,129.00,DE,abgeschlossen
1,3155,2026-03-31,hannes.vogel@example.com,Webcam HD,Video,3,79.00,DE,abgeschlossen
2,3140,2026-02-24,mia.lorenz@example.com,Beamer,Video,1,"1.299,00",DE,abgeschlossen
3,3179,2026-03-13,tina.jaeger@example.com,Laptopständer,Zubehör,3,24.95,DE,abgeschlossen
4,3016,2026-03-23,ida.neumann@example.com,Maus,Zubehör,3,"19,90",DE,abgeschlossen


In [2]:
# Schritt 2: Erste Datenqualitätsprüfung

print("=== DATENTYPEN ===")
print(df.dtypes)

print("\n=== FEHLENDE WERTE ===")
print(df.isna().sum())

print("\n=== EINDEUTIGE WERTE PRO SPALTE ===")
print(df.nunique())

print("\n=== VOLLSTÄNDIG DOPPELTE ZEILEN ===")
print(df.duplicated().sum())

=== DATENTYPEN ===
bestell_id     int64
datum            str
kunde_email      str
produkt          str
kategorie        str
menge          int64
preis_eur        str
land             str
status           str
dtype: object

=== FEHLENDE WERTE ===
bestell_id     0
datum          0
kunde_email    9
produkt        0
kategorie      6
menge          0
preis_eur      0
land           0
status         0
dtype: int64

=== EINDEUTIGE WERTE PRO SPALTE ===
bestell_id     209
datum          101
kunde_email     33
produkt         11
kategorie        5
menge            6
preis_eur       19
land             8
status           3
dtype: int64

=== VOLLSTÄNDIG DOPPELTE ZEILEN ===
8


In [3]:
# Schritt 3: Unterschiedliche Werte untersuchen

for spalte in ["land", "kategorie", "status", "menge", "preis_eur"]:
    print(f"\n=== {spalte.upper()} ===")
    print(df[spalte].value_counts(dropna=False))


=== LAND ===
land
DE             172
AT              21
CH               7
Deutschland      5
de               4
Schweiz          4
Österreich       3
at               2
Name: count, dtype: int64

=== KATEGORIE ===
kategorie
Zubehör     122
Audio        42
Video        23
Monitore     18
zubehör       7
NaN           6
Name: count, dtype: int64

=== STATUS ===
status
abgeschlossen    199
offen             12
storniert          7
Name: count, dtype: int64

=== MENGE ===
menge
1      58
2      57
3      56
4      44
0       2
180     1
Name: count, dtype: int64

=== PREIS_EUR ===
preis_eur
9.95        27
39.90       26
19.90       22
59.90       21
249.00      19
79.00       18
129.00      15
24.95       15
89.00       14
9,95         7
129,00       6
19,90        5
24,95        5
59,90        5
39,90        4
1.299,00     3
89,00        3
79,00        2
249,00       1
Name: count, dtype: int64


In [4]:
# Schritt 4: Datumsformate und doppelte Bestell-IDs prüfen

print("=== DATUMSFORMATE ===")
print(df["datum"].head(20).to_string(index=False))

print("\n=== DOPPELTE BESTELL-IDs ===")
doppelte_ids = df[df["bestell_id"].duplicated(keep=False)]

print("Betroffene Zeilen:", len(doppelte_ids))
print("Betroffene Bestell-IDs:", doppelte_ids["bestell_id"].nunique())

display(doppelte_ids.sort_values("bestell_id"))

=== DATUMSFORMATE ===
2026-02-21
2026-03-31
2026-02-24
2026-03-13
2026-03-23
2026-03-31
2026-02-23
2026-01-05
2026-01-16
2026-02-11
2026-02-12
2026-02-16
2026-02-21
2026-03-04
14.02.2026
2026-01-05
2026-03-12
2026-01-16
28.03.2026
2026-03-25

=== DOPPELTE BESTELL-IDs ===
Betroffene Zeilen: 18
Betroffene Bestell-IDs: 9


,bestell_id,datum,kunde_email,produkt,kategorie,menge,preis_eur,land,status
190,3005,2026-03-24,katja.wolf@example.com,Laptopständer,Zubehör,1,24.95,DE,abgeschlossen
114,3005,2026-03-24,katja.wolf@example.com,Laptopständer,Zubehör,1,24.95,DE,abgeschlossen
82,3042,2026-02-21,tom.keller@example.com,Headset,Audio,2,59.90,DE,abgeschlossen
12,3042,2026-02-21,tom.keller@example.com,Headset,Audio,2,59.90,DE,abgeschlossen
78,3052,2026-01-24,hannes.vogel@example.com,Tastatur,Zubehör,4,39.90,DE,abgeschlossen
107,3052,2026-01-24,hannes.vogel@example.com,Tastatur,Zubehör,4,39.90,DE,abgeschlossen
81,3077,2026-02-11,anna.berger@example.com,Monitor 27 Zoll,Monitore,1,249.00,DE,abgeschlossen
112,3077,2026-02-11,anna.berger@example.com,Monitor 27 Zoll,Monitore,3,249.00,DE,abgeschlossen
32,3097,2026-03-01,simon.gross@example.com,Headset,Audio,1,59.90,DE,abgeschlossen
40,3097,2026-03-01,simon.gross@example.com,Headset,Audio,1,59.90,DE,abgeschlossen


In [5]:
# Schritt 5: Datumsformate systematisch zählen

datum_iso = df["datum"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")
datum_de = df["datum"].str.fullmatch(r"\d{2}\.\d{2}\.\d{4}")

print("=== DATUMSFORMATE ===")
print("ISO-Format (JJJJ-MM-TT):", datum_iso.sum())
print("Deutsches Format (TT.MM.JJJJ):", datum_de.sum())
print("Andere Formate:", (~(datum_iso | datum_de)).sum())

print("\n=== UNBEKANNTE DATUMSFORMATE ===")
display(df.loc[~(datum_iso | datum_de), ["bestell_id", "datum"]])

=== DATUMSFORMATE ===
ISO-Format (JJJJ-MM-TT): 194
Deutsches Format (TT.MM.JJJJ): 24
Andere Formate: 0

=== UNBEKANNTE DATUMSFORMATE ===


,bestell_id,datum


## Woche 2 – Tag 1: Datenprofiling und Befundliste

### Ausgangslage
Die Datei `bestellungen_q1_roh.csv` enthält 218 Datensätze und 9 Spalten.
Ziel ist es, Datenqualitätsprobleme zu identifizieren, fachlich zu bewerten
und erst anschließend geeignete Bereinigungsregeln festzulegen.

### Befundliste vor der Bereinigung

| Nr. | Fehlerklasse / Auffälligkeit | Beleg | Fachliche Bedeutung |
|---|---|---|---|
| 1 | Fehlende Kunden-E-Mails | 9 fehlende Werte | Kundenkontakt und Zuordnung erschwert |
| 2 | Fehlende Produktkategorien | 6 fehlende Werte | Fehlerhafte Kategorieauswertungen möglich |
| 3 | Vollständig doppelte Datensätze | 8 Duplikate | Gefahr einer doppelten Umsatzberechnung |
| 4 | Mehrfach verwendete Bestell-IDs | 9 IDs, 18 betroffene Zeilen | Bestellpositionen oder Erfassungsfehler möglich |
| 5 | Uneinheitliche Länderbezeichnungen | 8 Schreibweisen für 3 Länder | Verfälschte Länderstatistiken |
| 6 | Uneinheitliche Kategoriebezeichnungen | 7-mal `zubehör` statt `Zubehör` | Getrennte Auswertung gleicher Kategorien |
| 7 | Ungültige Bestellmengen | 2 Datensätze mit Menge 0 | Keine reguläre verkaufte Stückzahl |
| 8 | Auffällige Bestellmenge | 1 Datensatz mit Menge 180 | Mögliche Großbestellung oder Eingabefehler |
| 9 | Unterschiedliche Preisformate | Punkt und Komma als Dezimaltrennzeichen | Fehler bei Umsatzberechnungen möglich |
| 10 | Unterschiedliche Datumsformate | 194 ISO, 24 deutsch | Einheitliche zeitliche Auswertung erschwert |

### Fachliche Besonderheiten

Nicht jede Auffälligkeit ist automatisch ein Fehler.

Beispielsweise kommt die Bestell-ID 3077 zweimal vor.
Beide Datensätze betreffen denselben Kunden und dasselbe Produkt,
unterscheiden sich jedoch in der Bestellmenge (1 und 3).

Eine automatische Löschung oder Zusammenführung wäre ohne weitere
Informationen nicht gerechtfertigt.

Auch die Bestellung mit Menge 180 könnte eine legitime Großbestellung sein.
Diese wird zunächst nur als auffällig gekennzeichnet.

### Grundsatz

Automatische Bereinigungen erfolgen nur bei eindeutig bestimmbaren
Format- oder Schreibfehlern. Fachlich mehrdeutige Fälle werden zunächst
markiert und gesondert geprüft.

## Aufgabe 2: Priorisierung nach Impact × Aufwand

Bewertungsskala:
- Impact: 1 = gering, 5 = sehr hoch
- Aufwand: 1 = gering, 5 = sehr hoch

Für die Reihenfolge wird der Impact ins Verhältnis zum Aufwand gesetzt.
Ein hoher Wert bedeutet einen hohen erwarteten Nutzen bei vergleichsweise geringem Aufwand.

| Priorität | Datenproblem | Impact | Aufwand | Prioritätswert (Impact / Aufwand) |
|---|---|---:|---:|---:|
| 1 | Unterschiedliche Preisformate | 5 | 1 | 5,0 |
| 2 | Uneinheitliche Länderbezeichnungen | 4 | 1 | 4,0 |
| 3 | Unterschiedliche Datumsformate | 4 | 1 | 4,0 |
| 4 | Uneinheitliche Kategoriebezeichnungen | 3 | 1 | 3,0 |
| 5 | Vollständig doppelte Datensätze | 5 | 3 | 1,7 |

### Begründung

**1. Preisformate:** Unterschiedliche Dezimal- und Tausendertrennzeichen
können Umsatzberechnungen verfälschen. Eine regelbasierte Umwandlung
ist technisch vergleichsweise einfach.

**2. Länderbezeichnungen:** Acht unterschiedliche Schreibweisen
für drei Länder führen zu ungenauen Länderstatistiken.
Die bekannten Varianten lassen sich eindeutig zuordnen.

**3. Datumsformate:** 24 Datensätze verwenden das deutsche Datumsformat.
Eine Vereinheitlichung ermöglicht zuverlässige zeitliche Auswertungen.

**4. Kategoriebezeichnungen:** Sieben Datensätze verwenden `zubehör`
statt `Zubehör`. Eine einheitliche Schreibweise verbessert
die Kategorieauswertung.

**5. Vollständig doppelte Datensätze:** Acht vollständig doppelte Zeilen
könnten zu einer doppelten Umsatzberechnung führen.
Da Bestellungen auch mehrere Positionen enthalten können,
muss vor einer Löschung die fachliche Bedeutung geprüft werden.

### Weitere wichtige Prüfungen

Fehlende E-Mails, fehlende Kategorien, ungültige Mengen und
mehrfach verwendete Bestell-IDs bleiben relevant.

Insbesondere die Bestell-ID 3077 und die Menge 180 werden
nicht automatisch korrigiert, da ihre fachliche Bedeutung
noch nicht eindeutig geklärt ist.

Die Priorisierung ist eine begründete Ersteinschätzung und
kann nach weiteren Prüfungen angepasst werden.

## Aufgabe 3: Bereinigungsplan vor der Umsetzung

### Ziel
Die Rohdaten sollen für zuverlässige Auswertungen vorbereitet werden.
Dabei werden eindeutige Formatfehler korrigiert und fachlich unklare
Datensätze nicht ohne Prüfung verändert.

| Nr. | Problem | Geplante Regel | Erwarteter Beleg |
|---|---|---|---|
| 1 | Unterschiedliche Preisformate | Deutsche und internationale Preisformate einheitlich in numerische Werte umwandeln | `preis_eur` ist numerisch; keine ungültigen Umwandlungen |
| 2 | Länderbezeichnungen | Deutschland → DE, de → DE, Österreich → AT, at → AT, Schweiz → CH | Nur noch DE, AT und CH vorhanden |
| 3 | Datumsformate | ISO- und deutsche Datumsangaben einheitlich als Datum interpretieren | 218 gültige Datumswerte; keine ungültigen Werte |
| 4 | Kategoriebezeichnungen | `zubehör` zu `Zubehör` vereinheitlichen | Keine Einträge mit `zubehör` mehr vorhanden |
| 5 | Vollständig doppelte Zeilen | Exakte Duplikate identifizieren und nur bei bestätigter technischer Doppelerfassung entfernen | Anzahl und betroffene Bestell-IDs dokumentiert |
| 6 | Fehlende Kunden-E-Mails | Fehlende Werte beibehalten; bekannte Platzhalter gesondert prüfen | Fehlende Werte und Platzhalter ausgewiesen |
| 7 | Fehlende Kategorien | Keine Kategorien ohne eindeutige Produktzuordnung ergänzen | Ungeklärte Kategorien dokumentiert |
| 8 | Menge 0 | Als ungültige Verkaufsmenge kennzeichnen; nicht automatisch ersetzen | 2 betroffene Datensätze ausgewiesen |
| 9 | Menge 180 | Als möglichen Ausreißer markieren | 1 auffälliger Datensatz ausgewiesen |
| 10 | Mehrfach verwendete Bestell-IDs | Identische und unterschiedliche Bestellpositionen getrennt untersuchen | 9 betroffene IDs nachvollziehbar dokumentiert |

### Fachliche Entscheidungen

**Dilemma 1: Doppelte Bestell-ID 3077**

Die Bestell-ID 3077 kommt zweimal vor. Beide Zeilen enthalten
denselben Kunden und dasselbe Produkt, aber unterschiedliche Mengen
(1 und 3).

Entscheidung: Beide Zeilen zunächst behalten und zur Prüfung markieren.

Begründung: Es könnte sich um zwei Bestellpositionen, eine
nachträgliche Mengenänderung oder einen Erfassungsfehler handeln.

Verworfene Alternative: Eine Zeile automatisch löschen oder
beide Mengen ungeprüft addieren.

**Dilemma 2: Auffällige Bestellmenge 180**

Eine Bestellung enthält die Menge 180.

Entscheidung: Datensatz zunächst behalten und als auffällig markieren.

Begründung: Eine ungewöhnlich hohe Menge kann eine legitime
Großbestellung sein.

Verworfene Alternative: Die Menge automatisch auf einen
vermeintlich normalen Wert reduzieren oder die Bestellung löschen.

### Qualitätssicherung

Nach der Umsetzung werden folgende Punkte geprüft:

- Anzahl der Datensätze vorher und nachher
- Anzahl der Duplikate
- Gültigkeit der Datumsangaben
- Einheitliche Preisformate
- Einheitliche Länder- und Kategoriebezeichnungen
- Auswirkungen auf die Umsatzberechnung
- Dokumentation aller ungeklärten Fälle

Die Rohdaten bleiben unverändert erhalten.

In [6]:
# Schritt 8: Arbeitskopie für die Datenbereinigung erstellen

df_clean = df.copy(deep=True)

print("=== AUSGANGSLAGE ===")
print("Rohdaten:", df.shape)
print("Arbeitskopie:", df_clean.shape)

# Prüfen, ob beide DataFrames zunächst identisch sind
pd.testing.assert_frame_equal(df, df_clean)

print("\n✅ Arbeitskopie erfolgreich erstellt.")
print("Die Rohdaten bleiben unverändert.")

=== AUSGANGSLAGE ===
Rohdaten: (218, 9)
Arbeitskopie: (218, 9)

✅ Arbeitskopie erfolgreich erstellt.
Die Rohdaten bleiben unverändert.


In [7]:
# Schritt 9: Preisformate vor der Bereinigung untersuchen

preise = df_clean["preis_eur"].astype("string").str.strip()

# Unterschiedliche Schreibweisen erkennen
format_de = preise.str.fullmatch(r"\d{1,3}(\.\d{3})+,\d{2}|\d+,\d{2}")
format_en = preise.str.fullmatch(r"\d+\.\d{2}")

print("=== PREISFORMATE ===")
print("Deutsches Format:", format_de.sum())
print("Format mit Dezimalpunkt:", format_en.sum())
print("Andere oder fehlende Formate:", (~(format_de | format_en)).sum())

print("\n=== UNBEKANNTE PREISFORMATE ===")
display(df_clean.loc[
    ~(format_de | format_en).fillna(False),
    ["bestell_id", "preis_eur"]
])

=== PREISFORMATE ===
Deutsches Format: 41
Format mit Dezimalpunkt: 177
Andere oder fehlende Formate: 0

=== UNBEKANNTE PREISFORMATE ===


,bestell_id,preis_eur


In [8]:
# Schritt 10: Preisformate vereinheitlichen

# Deutsche Schreibweise zuerst umwandeln
preise_bereinigt = preise.copy()

preise_bereinigt.loc[format_de] = (
    preise.loc[format_de]
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)

# Alle Preise in Zahlen umwandeln
df_clean["preis_eur"] = pd.to_numeric(
    preise_bereinigt,
    errors="raise"
)

print("=== PREISBEREINIGUNG ===")
print("Datentyp vorher:", df["preis_eur"].dtype)
print("Datentyp nachher:", df_clean["preis_eur"].dtype)

print("Deutsche Preisformate umgewandelt:", format_de.sum())
print("Ungültige Preise nach Bereinigung:", df_clean["preis_eur"].isna().sum())

# Beispiel: Tausendertrennzeichen kontrollieren
display(
    df_clean.loc[
        df["preis_eur"].astype("string").str.contains("1.299", regex=False),
        ["bestell_id", "preis_eur"]
    ]
)

=== PREISBEREINIGUNG ===
Datentyp vorher: str
Datentyp nachher: Float64
Deutsche Preisformate umgewandelt: 41
Ungültige Preise nach Bereinigung: 0


,bestell_id,preis_eur
2,3140,1299.0
70,3032,1299.0
74,3043,1299.0


In [9]:
# Schritt 11: Qualitätstests für die Preisbereinigung

# Alle Preise müssen numerisch und vorhanden sein
assert pd.api.types.is_numeric_dtype(df_clean["preis_eur"])
assert df_clean["preis_eur"].notna().sum() == 218

# Alle Preise müssen größer als 0 sein
assert (df_clean["preis_eur"] > 0).all()

# Beispiel mit Tausendertrennzeichen überprüfen
maske = df["preis_eur"].astype("string").eq("1.299,00")

assert maske.sum() == 3
assert (df_clean.loc[maske, "preis_eur"] == 1299.00).all()

print("✅ Alle Qualitätstests bestanden!")
print("218 Preise erfolgreich geprüft.")
print("3 Preise mit Tausendertrennzeichen korrekt umgewandelt.")

✅ Alle Qualitätstests bestanden!
218 Preise erfolgreich geprüft.
3 Preise mit Tausendertrennzeichen korrekt umgewandelt.


In [10]:
# Schritt 12: Länderbezeichnungen vereinheitlichen

land_mapping = {
    "DE": "DE",
    "Deutschland": "DE",
    "de": "DE",
    "AT": "AT",
    "Österreich": "AT",
    "at": "AT",
    "CH": "CH",
    "Schweiz": "CH"
}

# Vorher dokumentieren
print("=== LÄNDER VORHER ===")
print(df_clean["land"].value_counts())

# Bekannte Schreibweisen vereinheitlichen
df_clean["land"] = df_clean["land"].replace(land_mapping)

# Nachher dokumentieren
print("\n=== LÄNDER NACHHER ===")
print(df_clean["land"].value_counts())

# Qualitätstests
assert set(df_clean["land"].dropna().unique()) == {"DE", "AT", "CH"}
assert df_clean["land"].notna().sum() == 218

print("\n✅ Länderbezeichnungen erfolgreich vereinheitlicht!")

=== LÄNDER VORHER ===
land
DE             172
AT              21
CH               7
Deutschland      5
de               4
Schweiz          4
Österreich       3
at               2
Name: count, dtype: int64

=== LÄNDER NACHHER ===
land
DE    181
AT     26
CH     11
Name: count, dtype: int64

✅ Länderbezeichnungen erfolgreich vereinheitlicht!


In [11]:
# Schritt 13: Datumsformate vereinheitlichen

# Datumswerte getrennt nach Format interpretieren
datum_bereinigt = pd.Series(pd.NaT, index=df_clean.index, dtype="datetime64[ns]")

datum_bereinigt.loc[datum_iso] = pd.to_datetime(
    df_clean.loc[datum_iso, "datum"],
    format="%Y-%m-%d",
    errors="coerce"
)

datum_bereinigt.loc[datum_de] = pd.to_datetime(
    df_clean.loc[datum_de, "datum"],
    format="%d.%m.%Y",
    errors="coerce"
)

# Qualität vor der Übernahme prüfen
assert datum_bereinigt.notna().sum() == 218
assert datum_bereinigt.between("2026-01-01", "2026-03-31").all()

# Bereinigte Werte übernehmen
df_clean["datum"] = datum_bereinigt

print("=== DATUMSBEREINIGUNG ===")
print("Datentyp vorher:", df["datum"].dtype)
print("Datentyp nachher:", df_clean["datum"].dtype)

print("ISO-Datumsangaben:", datum_iso.sum())
print("Deutsche Datumsangaben:", datum_de.sum())
print("Ungültige Datumswerte:", df_clean["datum"].isna().sum())

print("\nFrühestes Datum:", df_clean["datum"].min())
print("Spätestes Datum:", df_clean["datum"].max())

print("\n✅ Datumsbereinigung erfolgreich!")

=== DATUMSBEREINIGUNG ===
Datentyp vorher: str
Datentyp nachher: datetime64[ns]
ISO-Datumsangaben: 194
Deutsche Datumsangaben: 24
Ungültige Datumswerte: 0

Frühestes Datum: 2026-01-02 00:00:00
Spätestes Datum: 2026-03-31 00:00:00

✅ Datumsbereinigung erfolgreich!


In [12]:
 # Schritt 14: Kategoriebezeichnungen vereinheitlichen

print("=== KATEGORIEN VORHER ===")
print(df_clean["kategorie"].value_counts(dropna=False))

# Bekannte Schreibweise korrigieren
df_clean["kategorie"] = df_clean["kategorie"].replace({
    "zubehör": "Zubehör"
})

print("\n=== KATEGORIEN NACHHER ===")
print(df_clean["kategorie"].value_counts(dropna=False))

# Qualitätstests
assert (df_clean["kategorie"] == "zubehör").sum() == 0
assert (df_clean["kategorie"] == "Zubehör").sum() == 129
assert df_clean["kategorie"].isna().sum() == 6

print("\n✅ Kategoriebezeichnungen erfolgreich vereinheitlicht!")
print("6 fehlende Kategorien bleiben vorerst zur Prüfung erhalten.")

=== KATEGORIEN VORHER ===
kategorie
Zubehör     122
Audio        42
Video        23
Monitore     18
zubehör       7
NaN           6
Name: count, dtype: int64

=== KATEGORIEN NACHHER ===
kategorie
Zubehör     129
Audio        42
Video        23
Monitore     18
NaN           6
Name: count, dtype: int64

✅ Kategoriebezeichnungen erfolgreich vereinheitlicht!
6 fehlende Kategorien bleiben vorerst zur Prüfung erhalten.


In [13]:
# Schritt 15: Fehlende Kategorien fachlich untersuchen

print("=== BESTELLUNGEN OHNE KATEGORIE ===")

fehlende_kategorien = df_clean[df_clean["kategorie"].isna()]

display(
    fehlende_kategorien[
        ["bestell_id", "produkt", "kategorie"]
    ]
)

print("\n=== BEKANNTE KATEGORIEN DIESER PRODUKTE ===")

betroffene_produkte = fehlende_kategorien["produkt"].unique()

vergleich = df_clean[
    df_clean["produkt"].isin(betroffene_produkte)
    & df_clean["kategorie"].notna()
]

display(
    vergleich.groupby(["produkt", "kategorie"])
    .size()
    .reset_index(name="anzahl")
    .sort_values(["produkt", "anzahl"], ascending=[True, False])
)

=== BESTELLUNGEN OHNE KATEGORIE ===


,bestell_id,produkt,kategorie
68,3139,Headset,NaN
129,3071,Dockingstation,NaN
186,3089,Monitor 27 Zoll,NaN
191,3188,Tastatur,NaN
197,3002,Dockingstation,NaN
206,3203,Monitor 27 Zoll,NaN



=== BEKANNTE KATEGORIEN DIESER PRODUKTE ===


,produkt,kategorie,anzahl
0,Dockingstation,Zubehör,19
1,Headset,Audio,25
2,Monitor 27 Zoll,Monitore,18
3,Tastatur,Zubehör,25


In [14]:
# Schritt 16: Fehlende Kategorien anhand vorhandener Daten ergänzen

# Nur Produkte mit genau einer bekannten Kategorie berücksichtigen
kategorie_anzahl = (
    df_clean.dropna(subset=["kategorie"])
    .groupby("produkt")["kategorie"]
    .nunique()
)

eindeutige_produkte = kategorie_anzahl[kategorie_anzahl == 1].index

# Zuordnungstabelle erstellen
kategorie_mapping = (
    df_clean[
        df_clean["produkt"].isin(eindeutige_produkte)
        & df_clean["kategorie"].notna()
    ]
    .drop_duplicates(subset=["produkt"])
    .set_index("produkt")["kategorie"]
)

# Fehlende Kategorien ergänzen
fehlend_vorher = df_clean["kategorie"].isna().sum()

df_clean["kategorie"] = df_clean["kategorie"].fillna(
    df_clean["produkt"].map(kategorie_mapping)
)

fehlend_nachher = df_clean["kategorie"].isna().sum()

print("=== KATEGORIEN ERGÄNZEN ===")
print("Fehlende Kategorien vorher:", fehlend_vorher)
print("Fehlende Kategorien nachher:", fehlend_nachher)
print("Ergänzte Kategorien:", fehlend_vorher - fehlend_nachher)

# Qualitätstests
assert fehlend_vorher == 6
assert fehlend_nachher == 0

print("\n✅ Alle sechs Kategorien anhand eindeutiger Zuordnungen ergänzt!")

=== KATEGORIEN ERGÄNZEN ===
Fehlende Kategorien vorher: 6
Fehlende Kategorien nachher: 0
Ergänzte Kategorien: 6

✅ Alle sechs Kategorien anhand eindeutiger Zuordnungen ergänzt!


In [15]:
# Schritt 17: Duplikate vor einer Entscheidung untersuchen

# Vollständig identische Datensätze
exakte_duplikate = df_clean[
    df_clean.duplicated(keep=False)
].sort_values("bestell_id")

print("=== EXAKTE DUPLIKATE ===")
print("Betroffene Zeilen:", len(exakte_duplikate))
print("Doppelte Zeilen über die erste hinaus:",
      df_clean.duplicated().sum())

display(exakte_duplikate)

# Bestell-ID 3077 gesondert untersuchen
print("\n=== FACHLICHER SONDERFALL: BESTELL-ID 3077 ===")

display(
    df_clean.loc[
        df_clean["bestell_id"] == 3077,
        ["bestell_id", "datum", "kunde_email",
         "produkt", "menge", "preis_eur", "status"]
    ]
)

=== EXAKTE DUPLIKATE ===
Betroffene Zeilen: 16
Doppelte Zeilen über die erste hinaus: 8


,bestell_id,datum,kunde_email,produkt,kategorie,menge,preis_eur,land,status
114,3005,2026-03-24,katja.wolf@example.com,Laptopständer,Zubehör,1,24.95,DE,abgeschlossen
190,3005,2026-03-24,katja.wolf@example.com,Laptopständer,Zubehör,1,24.95,DE,abgeschlossen
12,3042,2026-02-21,tom.keller@example.com,Headset,Audio,2,59.9,DE,abgeschlossen
82,3042,2026-02-21,tom.keller@example.com,Headset,Audio,2,59.9,DE,abgeschlossen
78,3052,2026-01-24,hannes.vogel@example.com,Tastatur,Zubehör,4,39.9,DE,abgeschlossen
107,3052,2026-01-24,hannes.vogel@example.com,Tastatur,Zubehör,4,39.9,DE,abgeschlossen
32,3097,2026-03-01,simon.gross@example.com,Headset,Audio,1,59.9,DE,abgeschlossen
40,3097,2026-03-01,simon.gross@example.com,Headset,Audio,1,59.9,DE,abgeschlossen
62,3138,2026-01-13,nils.schaefer@example.com,USB-C-Kabel,Zubehör,3,9.95,DE,abgeschlossen
182,3138,2026-01-13,nils.schaefer@example.com,USB-C-Kabel,Zubehör,3,9.95,DE,abgeschlossen



=== FACHLICHER SONDERFALL: BESTELL-ID 3077 ===


,bestell_id,datum,kunde_email,produkt,menge,preis_eur,status
81,3077,2026-02-11,anna.berger@example.com,Monitor 27 Zoll,1,249.0,abgeschlossen
112,3077,2026-02-11,anna.berger@example.com,Monitor 27 Zoll,3,249.0,abgeschlossen


In [16]:
# Schritt 18: Exakte Duplikate kontrolliert entfernen

zeilen_vorher = len(df_clean)

# Betroffene Bestell-IDs dokumentieren
entfernte_ids = df_clean.loc[
    df_clean.duplicated(),
    "bestell_id"
].tolist()

# Nur vollständig identische Zeilen entfernen
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

zeilen_nachher = len(df_clean)

print("=== DUPLIKATBEREINIGUNG ===")
print("Zeilen vorher:", zeilen_vorher)
print("Zeilen nachher:", zeilen_nachher)
print("Entfernte Duplikate:", zeilen_vorher - zeilen_nachher)
print("Betroffene Bestell-IDs:", entfernte_ids)

# Qualitätstests
assert zeilen_vorher == 218
assert zeilen_nachher == 210
assert df_clean.duplicated().sum() == 0

# Sonderfall 3077 muss vollständig erhalten bleiben
assert (df_clean["bestell_id"] == 3077).sum() == 2
assert set(
    df_clean.loc[df_clean["bestell_id"] == 3077, "menge"]
) == {1, 3}

print("\n✅ Exakte Duplikate entfernt!")
print("✅ Bestell-ID 3077 mit beiden Mengen erhalten!")

=== DUPLIKATBEREINIGUNG ===
Zeilen vorher: 218
Zeilen nachher: 210
Entfernte Duplikate: 8
Betroffene Bestell-IDs: [3155, 3097, 3042, 3052, 3160, 3138, 3005, 3173]

✅ Exakte Duplikate entfernt!
✅ Bestell-ID 3077 mit beiden Mengen erhalten!


In [17]:
# Schritt 19: Auffällige Bestellmengen untersuchen

print("=== BESTELLMENGEN ===")
print(df_clean["menge"].value_counts().sort_index())

# Ungültige Mengen: 0 oder negativ
ungueltige_mengen = df_clean[df_clean["menge"] <= 0]

print("\n=== UNGÜLTIGE MENGEN ===")
print("Betroffene Datensätze:", len(ungueltige_mengen))

display(ungueltige_mengen)

# Auffällig hohe Mengen: über 100
hohe_mengen = df_clean[df_clean["menge"] > 100]

print("\n=== AUFFÄLLIG HOHE MENGEN ===")
print("Betroffene Datensätze:", len(hohe_mengen))

display(hohe_mengen)

=== BESTELLMENGEN ===
menge
0       2
1      56
2      55
3      53
4      43
180     1
Name: count, dtype: int64

=== UNGÜLTIGE MENGEN ===
Betroffene Datensätze: 2


,bestell_id,datum,kunde_email,produkt,kategorie,menge,preis_eur,land,status
102,3034,2026-01-24,mia.lorenz@example.com,USB-C-Kabel,Zubehör,0,9.95,DE,abgeschlossen
206,3015,2026-01-27,timo.frank@example.com,Webcam HD,Video,0,79.0,DE,abgeschlossen



=== AUFFÄLLIG HOHE MENGEN ===
Betroffene Datensätze: 1


,bestell_id,datum,kunde_email,produkt,kategorie,menge,preis_eur,land,status
208,3180,2026-02-13,timo.frank@example.com,USB-C-Kabel,Zubehör,180,9.95,DE,abgeschlossen


In [18]:
# Schritt 20: Auffällige Mengen kennzeichnen

df_clean["menge_pruefung"] = "OK"

# Ungültige Mengen markieren
df_clean.loc[
    df_clean["menge"] <= 0,
    "menge_pruefung"
] = "Ungültige Menge – prüfen"

# Ungewöhnlich hohe Mengen markieren
df_clean.loc[
    df_clean["menge"] > 100,
    "menge_pruefung"
] = "Hohe Menge – prüfen"

print("=== ERGEBNIS DER MENGENPRÜFUNG ===")
print(df_clean["menge_pruefung"].value_counts())

print("\n=== MARKIERTE BESTELLUNGEN ===")

display(
    df_clean.loc[
        df_clean["menge_pruefung"] != "OK",
        ["bestell_id", "produkt", "menge", "status", "menge_pruefung"]
    ]
)

# Qualitätstests
assert (df_clean["menge_pruefung"] == "Ungültige Menge – prüfen").sum() == 2
assert (df_clean["menge_pruefung"] == "Hohe Menge – prüfen").sum() == 1
assert len(df_clean) == 210

print("\n✅ Auffällige Mengen erfolgreich gekennzeichnet!")
print("✅ Keine Datensätze entfernt oder Mengen verändert!")

=== ERGEBNIS DER MENGENPRÜFUNG ===
menge_pruefung
OK                          207
Ungültige Menge – prüfen      2
Hohe Menge – prüfen           1
Name: count, dtype: int64

=== MARKIERTE BESTELLUNGEN ===


,bestell_id,produkt,menge,status,menge_pruefung
102,3034,USB-C-Kabel,0,abgeschlossen,Ungültige Menge – prüfen
206,3015,Webcam HD,0,abgeschlossen,Ungültige Menge – prüfen
208,3180,USB-C-Kabel,180,abgeschlossen,Hohe Menge – prüfen



✅ Auffällige Mengen erfolgreich gekennzeichnet!
✅ Keine Datensätze entfernt oder Mengen verändert!


In [19]:
# Schritt 21: Fehlende und ungültige Kunden-E-Mails untersuchen

emails = df_clean["kunde_email"].astype("string").str.strip()

# Fehlende Werte
fehlende_emails = emails.isna() | emails.eq("")

# Einfache Prüfung auf E-Mail-Struktur
gueltige_emails = emails.str.fullmatch(
    r"[^@\s]+@[^@\s]+\.[^@\s]+"
).fillna(False)

# Vorhandene, aber ungültige Einträge
ungueltige_emails = ~fehlende_emails & ~gueltige_emails

print("=== E-MAIL-QUALITÄT ===")
print("Fehlende E-Mails:", fehlende_emails.sum())
print("Ungültige Einträge:", ungueltige_emails.sum())
print("Gültiges E-Mail-Format:", gueltige_emails.sum())

print("\n=== BETROFFENE BESTELLUNGEN ===")

display(
    df_clean.loc[
        fehlende_emails | ungueltige_emails,
        ["bestell_id", "kunde_email", "produkt", "status"]
    ]
)

=== E-MAIL-QUALITÄT ===
Fehlende E-Mails: 9
Ungültige Einträge: 3
Gültiges E-Mail-Format: 198

=== BETROFFENE BESTELLUNGEN ===


,bestell_id,kunde_email,produkt,status
0,3101,kontakt-fehlt,Dockingstation,abgeschlossen
5,3171,NaN,Headset,abgeschlossen
30,3056,NaN,Maus,abgeschlossen
36,3137,NaN,USB-C-Kabel,abgeschlossen
52,3209,NaN,Headset,abgeschlossen
55,3070,NaN,Maus,abgeschlossen
59,3006,NaN,Maus,abgeschlossen
61,3057,NaN,Dockingstation,abgeschlossen
114,3125,NaN,Lautsprecher,abgeschlossen
127,3065,NaN,Headset,abgeschlossen


In [20]:
# Schritt 22: E-Mail-Qualität kennzeichnen

df_clean["email_pruefung"] = "OK"

# Fehlende E-Mail-Adressen markieren
df_clean.loc[
    fehlende_emails,
    "email_pruefung"
] = "E-Mail fehlt"

# Ungültige E-Mail-Adressen markieren
df_clean.loc[
    ungueltige_emails,
    "email_pruefung"
] = "Ungültiges E-Mail-Format"

print("=== E-MAIL-QUALITÄT NACH KENNZEICHNUNG ===")
print(df_clean["email_pruefung"].value_counts())

# Qualitätstests
assert (df_clean["email_pruefung"] == "OK").sum() == 198
assert (df_clean["email_pruefung"] == "E-Mail fehlt").sum() == 9
assert (df_clean["email_pruefung"] == "Ungültiges E-Mail-Format").sum() == 3
assert len(df_clean) == 210

print("\n✅ E-Mail-Qualität erfolgreich gekennzeichnet!")
print("✅ Keine E-Mail-Adressen erfunden oder verändert!")

=== E-MAIL-QUALITÄT NACH KENNZEICHNUNG ===
email_pruefung
OK                          198
E-Mail fehlt                  9
Ungültiges E-Mail-Format      3
Name: count, dtype: int64

✅ E-Mail-Qualität erfolgreich gekennzeichnet!
✅ Keine E-Mail-Adressen erfunden oder verändert!


In [21]:
# Schritt 23: Q1-Umsatz und Auswirkungen der Bereinigung

# Ursprüngliche Preise mit derselben geprüften Formatregel umwandeln
roh_preise = df["preis_eur"].astype("string").str.strip()

roh_de = roh_preise.str.fullmatch(
    r"\d{1,3}(\.\d{3})+,\d{2}|\d+,\d{2}"
).fillna(False)

roh_preise.loc[roh_de] = (
    roh_preise.loc[roh_de]
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)

roh_preise = pd.to_numeric(roh_preise, errors="raise")

# Umsatz vor der Duplikatbereinigung
umsatz_vorher = (
    df.loc[df["status"] == "abgeschlossen", "menge"]
    * roh_preise.loc[df["status"] == "abgeschlossen"]
).sum()

# Umsatz nach der Bereinigung
df_clean["umsatz_eur"] = (
    df_clean["menge"] * df_clean["preis_eur"]
)

abgeschlossen = df_clean["status"] == "abgeschlossen"

umsatz_nachher = df_clean.loc[
    abgeschlossen, "umsatz_eur"
].sum()

# Großbestellung gesondert betrachten
grossbestellung = (
    abgeschlossen
    & (df_clean["bestell_id"] == 3180)
)

umsatz_grossbestellung = df_clean.loc[
    grossbestellung, "umsatz_eur"
].sum()

print("=== Q1-UMSATZREPORT ===")
print(f"Vor Bereinigung: {umsatz_vorher:,.2f} €")
print(f"Nach Bereinigung: {umsatz_nachher:,.2f} €")
print(f"Veränderung: {umsatz_nachher - umsatz_vorher:,.2f} €")

print("\n=== GROSSBESTELLUNG 3180 ===")
print(f"Bestellwert: {umsatz_grossbestellung:,.2f} €")
print(
    f"Q1 ohne Großbestellung: "
    f"{umsatz_nachher - umsatz_grossbestellung:,.2f} €"
)

# Qualitätstests
assert len(df_clean) == 210
assert grossbestellung.sum() == 1
assert abs(umsatz_grossbestellung - 1791.00) < 0.01

print("\n✅ Q1-Auswertung erfolgreich berechnet!")

=== Q1-UMSATZREPORT ===
Vor Bereinigung: 38,661.80 €
Nach Bereinigung: 37,263.80 €
Veränderung: -1,398.00 €

=== GROSSBESTELLUNG 3180 ===
Bestellwert: 1,791.00 €
Q1 ohne Großbestellung: 35,472.80 €

✅ Q1-Auswertung erfolgreich berechnet!


Aufgabe 4: Fachliche Entscheidungen und Dilemmas

Dilemma 1: Doppelte Bestellungen und Bestell-ID 3077

Befund:

In den ursprünglichen 218 Datensätzen wurden acht vollständig identische Duplikate festgestellt.

Zusätzlich enthält die Bestell-ID 3077 zwei Einträge desselben Kunden für dasselbe Produkt am selben Datum, jedoch mit unterschiedlichen Mengen (1 und 3).

Regel:

Vollständig identische Datensätze werden für die bereinigte Auswertung als technische Duplikate behandelt. Jeweils ein Datensatz bleibt erhalten.

Einträge mit derselben Bestell-ID, aber unterschiedlichen Mengen werden nicht automatisch zusammengeführt oder gelöscht.

Begründung:

Die acht identischen Datensatzpaare könnten durch eine doppelte Datenübertragung entstanden sein. Ihre mehrfache Berücksichtigung würde den ausgewiesenen Bestellwert möglicherweise verfälschen.

Bei Bestell-ID 3077 ist dagegen nicht eindeutig erkennbar, ob zwei Bestellpositionen, eine Mengenänderung oder ein Erfassungsfehler vorliegen.

Verworfene Alternativen:

* Alle mehrfach vorkommenden Bestell-IDs pauschal löschen.
* Die Mengen 1 und 3 automatisch addieren.
* Einen der beiden Einträge von Bestell-ID 3077 ohne weitere Prüfung entfernen.

Delta:

* Datensätze: 218 → 210
* Exakte überzählige Duplikate: 8 → 0
* Bestell-ID 3077: Beide Einträge bleiben erhalten.
* Rechnerischer Bestellwert abgeschlossener Bestellungen: 38.661,80 € → 37.263,80 €
* Differenz: −1.398,00 €

Einschränkung:

Ob die acht identischen Einträge tatsächlich technische Duplikate sind, müsste im Quellsystem bestätigt werden.

Auch die Bestell-ID 3077 bleibt fachlich ungeklärt.

Dilemma 2: Auffällige Bestellmengen

Befund:

Zwei abgeschlossene Bestellungen enthalten die Menge 0:

* Bestell-ID 3034: USB-C-Kabel
* Bestell-ID 3015: Webcam HD

Eine weitere abgeschlossene Bestellung enthält eine ungewöhnlich hohe Menge:

* Bestell-ID 3180: 180 USB-C-Kabel zu jeweils 9,95 €

Regel:

Bestellmengen von 0 werden als ungültig gekennzeichnet. Mengen über 100 werden als auffällig markiert.

Die ursprünglichen Mengen bleiben erhalten, bis eine fachliche Prüfung erfolgt.

Begründung:

Eine abgeschlossene Bestellung mit Menge 0 ist erklärungsbedürftig. Ohne weitere Informationen kann die tatsächliche Menge jedoch nicht bestimmt werden.

Die Menge 180 könnte eine legitime Großbestellung sein. Sie darf deshalb nicht automatisch als Fehler behandelt werden.

Verworfene Alternativen:

* Menge 0 automatisch durch Menge 1 ersetzen.
* Bestellungen mit Menge 0 ohne Prüfung löschen.
* Die Menge 180 auf einen niedrigeren Wert reduzieren.
* Die Großbestellung vollständig aus den bereinigten Daten entfernen.

Delta:

* Zwei Bestellungen als „Ungültige Menge – prüfen“ markiert.
* Eine Bestellung als „Hohe Menge – prüfen“ markiert.
* Keine Mengen verändert.
* Keine Datensätze aufgrund ihrer Mengen entfernt.
* Rechnerischer Bestellwert der Großbestellung: 1.791,00 €
* Q1-Bestellwert einschließlich Großbestellung: 37.263,80 €
* Q1-Bestellwert ohne Großbestellung: 35.472,80 €

Einschränkung:

Der Grenzwert von 100 Stück ist eine vorläufige Prüfregel und keine allgemeingültige Grenze für fehlerhafte Bestellungen.

Die Großbestellung sollte anhand der ursprünglichen Bestellung überprüft werden.

Fazit

Die Bereinigung unterscheidet zwischen eindeutig korrigierbaren Formatfehlern und fachlich unklaren Geschäftsvorgängen.

Während Preis-, Datums- und Länderformate vereinheitlicht wurden, bleiben ungeklärte Bestellungen nachvollziehbar gekennzeichnet.

Dadurch werden die Daten für Auswertungen besser nutzbar, ohne unbekannte Informationen zu erfinden oder auffällige Geschäftsvorgänge ungeprüft zu verändern.

In [22]:
# Schritt 25: Vorher-Nachher-Vergleich

vergleich_qualitaet = pd.DataFrame({
    "Kennzahl": [
        "Datensätze",
        "Überzählige exakte Duplikate",
        "Unterschiedliche Länderbezeichnungen",
        "Fehlende Kategorien",
        "Nicht numerische Preisspalte",
        "Nicht konvertierte Datumsspalte",
        "Ungültige Bestellmengen",
        "Auffällig hohe Bestellmengen"
    ],
    "Vorher": [
        len(df),
        df.duplicated().sum(),
        df["land"].nunique(),
        df["kategorie"].isna().sum(),
        not pd.api.types.is_numeric_dtype(df["preis_eur"]),
        not pd.api.types.is_datetime64_any_dtype(df["datum"]),
        (df["menge"] <= 0).sum(),
        (df["menge"] > 100).sum()
    ],
    "Nachher": [
        len(df_clean),
        df_clean.duplicated().sum(),
        df_clean["land"].nunique(),
        df_clean["kategorie"].isna().sum(),
        not pd.api.types.is_numeric_dtype(df_clean["preis_eur"]),
        not pd.api.types.is_datetime64_any_dtype(df_clean["datum"]),
        (df_clean["menge"] <= 0).sum(),
        (df_clean["menge"] > 100).sum()
    ]
})

print("=== DATENQUALITÄT VORHER / NACHHER ===")

display(vergleich_qualitaet)

# Abschließende grundlegende Qualitätstests
assert len(df) == 218
assert len(df_clean) == 210
assert df_clean.duplicated().sum() == 0
assert df_clean["kategorie"].isna().sum() == 0
assert df_clean["land"].nunique() == 3
assert df_clean["datum"].notna().all()
assert df_clean["preis_eur"].notna().all()

print("\n✅ Vorher-Nachher-Vergleich erfolgreich erstellt!")
print("✅ Alle grundlegenden Qualitätstests bestanden!")

=== DATENQUALITÄT VORHER / NACHHER ===


,Kennzahl,Vorher,Nachher
0,Datensätze,218,210
1,Überzählige exakte Duplikate,8,0
2,Unterschiedliche Länderbezeichnungen,8,3
3,Fehlende Kategorien,6,0
4,Nicht numerische Preisspalte,True,False
5,Nicht konvertierte Datumsspalte,True,False
6,Ungültige Bestellmengen,2,2
7,Auffällig hohe Bestellmengen,1,1



✅ Vorher-Nachher-Vergleich erfolgreich erstellt!
✅ Alle grundlegenden Qualitätstests bestanden!


In [23]:
# Schritt 26: Bereinigte Daten als Parquet exportieren

# Speicherort festlegen
export_datei = ordner / "bestellungen_q1_clean.parquet"

# Index zurücksetzen und Daten exportieren
df_export = df_clean.reset_index(drop=True)

df_export.to_parquet(
    export_datei,
    index=False
)

# Exportierte Datei erneut einlesen
df_kontrolle = pd.read_parquet(export_datei)

# Überprüfen, ob alle Daten korrekt gespeichert wurden
pd.testing.assert_frame_equal(
    df_export,
    df_kontrolle
)

print("=== PARQUET-EXPORT ===")
print("Speicherort:", export_datei)
print("Datensätze:", len(df_kontrolle))
print("Spalten:", len(df_kontrolle.columns))

print("\n✅ Parquet-Datei erfolgreich erstellt!")
print("✅ Rücklesetest bestanden – alle Daten stimmen überein!")

=== PARQUET-EXPORT ===
Speicherort: /Users/saschagolparvari/Desktop/Modul 5/Aufgaben/Woche_2/bestellungen_q1_clean.parquet
Datensätze: 210
Spalten: 12

✅ Parquet-Datei erfolgreich erstellt!
✅ Rücklesetest bestanden – alle Daten stimmen überein!


PII-Notiz – Datenschutz und personenbezogene Daten

1. Welche personenbezogenen Daten sind vorhanden?

Der Datensatz enthält die Spalte kunde_email.

E-Mail-Adressen können einzelne Personen identifizierbar machen und gelten deshalb grundsätzlich als personenbezogene Daten.

2. Welche Risiken bestehen?

Bei einer ungeschützten Weitergabe des Datensatzes könnten E-Mail-Adressen offengelegt werden.

Besonders kritisch wäre die Veröffentlichung auf GitHub oder die Verwendung echter Kundendaten in öffentlich zugänglichen Dashboards.

3. Wie können wir die Daten schützen?

Für spätere Auswertungen sollten folgende Maßnahmen berücksichtigt werden:

* E-Mail-Adressen aus öffentlichen Reports entfernen.
* Personenbezogene Daten nur verarbeiten, wenn sie für den jeweiligen Zweck erforderlich sind.
* Bei Bedarf pseudonymisierte Kundenkennungen verwenden.
* Den Zugriff auf Rohdaten beschränken.
* Für öffentliche Projekte ausschließlich synthetische oder ausreichend anonymisierte Beispieldaten verwenden.

4. Entscheidung für unser Projekt

Die E-Mail-Adressen bleiben in der lokalen Arbeitskopie erhalten, da sie für die Untersuchung von Datenqualitätsproblemen relevant sein können.

Für ein späteres Dashboard sollen die personenbezogenen Daten jedoch entfernt oder durch geeignete Kennungen ersetzt werden.

Vor einer Veröffentlichung auf GitHub muss geprüft werden, dass keine echten personenbezogenen Daten enthalten sind.

5. Offene Frage für Mittwoch

Wie können wir Kunden über mehrere Bestellungen hinweg wiedererkennen, ohne ihre E-Mail-Adressen direkt in unseren Analysedaten speichern zu müssen?

Eine mögliche Lösung wäre die Verwendung pseudonymisierter Kunden-IDs. Dabei müsste insbesondere geprüft werden, wie die Zuordnung geschützt wird.

In [24]:
# Schritt 28: Abschließende Qualitätskontrolle

print("=== FINALE QUALITÄTSKONTROLLE ===")

# Rohdaten und bereinigte Daten vergleichen
assert len(df) == 218
assert len(df_clean) == 210

# Bereinigungen überprüfen
assert df_clean.duplicated().sum() == 0
assert df_clean["land"].nunique() == 3
assert df_clean["kategorie"].isna().sum() == 0
assert df_clean["preis_eur"].notna().all()
assert df_clean["datum"].notna().all()

# Fachlich ungeklärte Fälle müssen erhalten bleiben
assert (df_clean["menge"] <= 0).sum() == 2
assert (df_clean["menge"] > 100).sum() == 1

# Export überprüfen
assert export_datei.exists()

pd.testing.assert_frame_equal(
    df_clean.reset_index(drop=True),
    pd.read_parquet(export_datei)
)

print("Rohdaten:", len(df))
print("Bereinigte Datensätze:", len(df_clean))
print("Entfernte Duplikate:", len(df) - len(df_clean))
print("Verbleibende Länder:", sorted(df_clean["land"].unique()))
print("Fehlende Kategorien:", df_clean["kategorie"].isna().sum())
print("Fachlich zu prüfende Mengen:", (df_clean["menge"] <= 0).sum() + (df_clean["menge"] > 100).sum())

print("\n✅ Alle finalen Qualitätstests bestanden!")
print("✅ Parquet-Export erfolgreich überprüft!")
print("✅ Woche 2 – Tag 1 technisch abgeschlossen!")

=== FINALE QUALITÄTSKONTROLLE ===
Rohdaten: 218
Bereinigte Datensätze: 210
Entfernte Duplikate: 8
Verbleibende Länder: ['AT', 'CH', 'DE']
Fehlende Kategorien: 0
Fachlich zu prüfende Mengen: 3

✅ Alle finalen Qualitätstests bestanden!
✅ Parquet-Export erfolgreich überprüft!
✅ Woche 2 – Tag 1 technisch abgeschlossen!
